# Imersão Alura Santander — Nivelamento IA 2026
## Semana 02 — Python como infraestrutura de IA

Este notebook acompanha a live da Semana 02. Cada seção conecta um conceito de Python com um padrão real usado em aplicações de IA em produção.

> **Como usar:** execute as células em ordem. As células de texto explicam o *porquê*. As células de código mostram o *como*.

---
## ⚙️ Setup — instalação e autenticação

Antes de qualquer coisa, precisamos instalar a biblioteca do Groq e configurar a chave de API.

**Por que o Groq?**  
É uma plataforma gratuita (14.400 requests/dia no plano free) que roda o modelo **Llama 3.3 70B** com resposta em menos de 1 segundo. Usamos ele ao longo de toda a live.

> Crie sua chave em [console.groq.com](https://console.groq.com) → API Keys. No Colab, salve como secret com o nome `GROQ_API_KEY`.

In [ ]:
#!pip install groq

In [ ]:
from groq import Groq
from google.colab import userdata

api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=api_key)
print("Cliente Groq pronto!")

Cliente Groq pronto!


---
## 1. A função `type()` — por que ela importa para IA

### Conceito
Quando você aprende Python, `type()` parece básica demais. Mas ela se torna essencial quando você começa a trabalhar com APIs de IA — porque as respostas que você recebe **não são strings simples**. São objetos complexos com vários níveis aninhados.

### O problema real
Se você tentar usar `resposta` diretamente como se fosse um texto, vai receber um erro. Precisa navegar pelos atributos corretos até chegar na string.

### Analogia
É como receber uma caixa postal: o envelope não é a carta. Você precisa abrir o envelope → tirar a carta → ler o conteúdo. `type()` é sua ferramenta para entender o que está em cada nível da caixa.

In [ ]:
# Fazendo a primeira chamada ao Groq
resposta = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Qual a capital do Brasil?"}]
)

### Inspecionando os tipos nível por nível

Veja que `resposta` não é uma string — é um objeto com vários atributos. Precisamos "desembrulhar" até chegar no texto.

In [ ]:
# Imprimindo o objeto bruto — parece assustador, mas é só uma estrutura organizada
print(resposta)

ChatCompletion(id='chatcmpl-792042f5-d52c-4ef6-9a10-7998db165120', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='A capital do Brasil é Brasília. Brasília é uma cidade localizada no Distrito Federal e foi inaugurada em 21 de abril de 1960, substituindo o Rio de Janeiro como capital do país. Ela foi planejada e projetada pelo urbanista Lúcio Costa e pelo arquiteto Oscar Niemeyer, e é conhecida por sua arquitetura moderna e inovadora.\n\nBrasília é a sede do governo federal brasileiro e abriga muitos órgãos públicos, incluindo o Palácio do Planalto, o Congresso Nacional e o Supremo Tribunal Federal. A cidade também é um importante centro cultural e econômico do país, com uma população de mais de 3 milhões de habitantes.\n\nAlguns pontos turísticos famosos de Brasília incluem:\n\n* A Praça dos Três Poderes, que abriga o Palácio do Planalto, o Congresso Nacional e o Supremo Tribunal Federal\n* O Museu Nacional, que abriga uma grande col

In [ ]:
# Inspecionando os tipos em cada nível
print("Tipo da resposta completa:")
print(type(resposta))
print()

Tipo da resposta completa:
<class 'groq.types.chat.chat_completion.ChatCompletion'>



In [ ]:
print("Tipo de resposta.choices:")
print(type(resposta.choices))
print()

Tipo de resposta.choices:
<class 'list'>



In [ ]:
print("Tipo de resposta.choices[0]:")
print(type(resposta.choices[0]))
print()

Tipo de resposta.choices[0]:
<class 'groq.types.chat.chat_completion.Choice'>



In [ ]:
print("Tipo de resposta.choices[0].message:")
print(type(resposta.choices[0].message))
print()

Tipo de resposta.choices[0].message:
<class 'groq.types.chat.chat_completion_message.ChatCompletionMessage'>



In [ ]:
print("Tipo de resposta.choices[0].message.content:")
print(type(resposta.choices[0].message.content))
print()

Tipo de resposta.choices[0].message.content:
<class 'str'>



**Conclusão:** passamos por 4 objetos diferentes antes de chegar na `str`. Cada nível tem seus próprios atributos — por exemplo, `resposta.usage` mostra quantos tokens foram consumidos na chamada.

In [ ]:
# Só aqui chegamos numa string de fato
texto = resposta.choices[0].message.content
print("Conteúdo final:", texto)

# Bônus: tokens consumidos
print(f"\nTokens usados: {resposta.usage.total_tokens}")

Conteúdo final: A capital do Brasil é Brasília. Brasília é uma cidade localizada no Distrito Federal e foi inaugurada em 21 de abril de 1960, substituindo o Rio de Janeiro como capital do país. Ela foi planejada e projetada pelo urbanista Lúcio Costa e pelo arquiteto Oscar Niemeyer, e é conhecida por sua arquitetura moderna e inovadora.

Brasília é a sede do governo federal brasileiro e abriga muitos órgãos públicos, incluindo o Palácio do Planalto, o Congresso Nacional e o Supremo Tribunal Federal. A cidade também é um importante centro cultural e econômico do país, com uma população de mais de 3 milhões de habitantes.

Alguns pontos turísticos famosos de Brasília incluem:

* A Praça dos Três Poderes, que abriga o Palácio do Planalto, o Congresso Nacional e o Supremo Tribunal Federal
* O Museu Nacional, que abriga uma grande coleção de arte e artefatos brasileiros
* A Catedral Metropolitana, uma igreja católica projetada por Oscar Niemeyer
* O Estádio Nacional, um dos maiores estádios

---
## 2. f-string — a base de todo prompt dinâmico

### Conceito
Prompts em produção **nunca são textos fixos**. Eles mudam de acordo com o usuário, o contexto, o idioma, o histórico da conversa. A f-string é a ferramenta Python que permite isso.

### Conexão com IA
Na Semana 06 você vai encontrar o `PromptTemplate` do LangChain — ele é exatamente uma f-string com validação automática de variáveis. Aprender f-string agora é aprender o conceito por trás de todos os frameworks de IA.

### Técnico
Uma f-string é avaliada em tempo de execução: `f"Ola {nome}"` substitui `{nome}` pelo valor da variável no momento em que a linha é executada. Isso permite criar prompts personalizados para cada usuário.

In [ ]:
def prompt_bio(nome, idade, area):
    return (
        f"Crie uma bio de {nome}.\n"
        f"Idade: {idade}.\n"
        f"Especialidade: {area}.\n"
        "Tom: atrativo. 2 parágrafos."
    )

In [ ]:
nome  = input("Qual seu nome? ")
idade = int(input("Quantos anos? "))
area  = input("Área de interesse: ")

print(f"Ola, {nome}! Voce tem {idade} anos.")
print(f"Foco: {area}")

Qual seu nome? Elaine Rocha
Quantos anos? 38
Área de interesse: Inteligencia Artificial com foco em geração de imagens realistas
Ola, Elaine Rocha! Voce tem 38 anos.
Foco: Inteligencia Artificial com foco em geração de imagens realistas


Antes de enviar para a API, veja como ficou o prompt montado. Repare que é uma string comum — a mágica já aconteceu na f-string.

In [ ]:
prompt_gerado = prompt_bio(nome, idade, area)
print(prompt_gerado)

Crie uma bio de Elaine Rocha.
Idade: 38.
Especialidade: Inteligencia Artificial com foco em geração de imagens realistas.
Tom: atrativo. 2 parágrafos.


In [ ]:
# Agora enviamos o prompt dinâmico para o modelo
resposta = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt_gerado}]
)

print(resposta.choices[0].message.content)

**Conheça Elaine Rocha**
Elaine Rocha é uma especialista em Inteligência Artificial com apenas 38 anos, mas já faz ondas no mundo da tecnologia com sua abordagem inovadora e focada em geração de imagens realistas. Com um estilo atraente e uma personalidade cativante, Elaine se destaca como uma das principais referências em sua área de atuação. Sua paixão pela Inteligência Artificial começou desde cedo, e ao longo dos anos, ela desenvolveu uma habilidade excepcional em criar soluções inovadoras e eficazes que mudam a forma como as pessoas interagem com a tecnologia.

**Uma Vida Dedicada à Inovação**
Com uma carreira marcada por realizações impressionantes, Elaine Rocha é sinônimo de inovação e criatividade. Seu foco em geração de imagens realistas a levou a desenvolver projetos revolucionários que estão mudando a forma como as empresas e os indivíduos criam e interagem com o conteúdo visual. Com uma abordagem colaborativa e um estilo de liderança atraente, Elaine inspira equipes e parce

---
## 3. Condicionais e loops — a lógica que move agentes

### Conceito
Um agente de IA não chama o modelo para tudo — isso seria lento e caro. Antes de qualquer chamada, ele **classifica a intenção** com lógica Python simples e decide qual caminho seguir.

### Padrão de produção: roteador de intenção
O `if/elif/else` aqui funciona como um roteador: analisa a mensagem do usuário e decide qual agente especializado deve receber a chamada. É o mesmo padrão que empresas como Nubank, iFood e Mercado Livre usam em seus bots de atendimento.

### Por que não chamar o modelo direto para classificar?
Poderia — e às vezes vale a pena. Mas para classificações simples baseadas em palavras-chave, o `if/elif` é **instantâneo e gratuito**. O modelo entra só quando precisa gerar linguagem natural.

In [ ]:
# if/elif/else — classifica a intenção antes de chamar o modelo
def classificar(msg):
    msg = msg.lower()
    if "cancel" in msg or "reembolso" in msg:
        return "RETENCAO"
    elif "cobr" in msg or "fatura" in msg:
        return "FINANCEIRO"
    elif "trav" in msg or "erro" in msg:
        return "SUPORTE"
    else:
        return "OUTROS"

# f-string monta o prompt com o contexto correto para cada categoria
def responder(msg, categoria):
    prompt = (
        f"Voce e um agente de suporte da categoria {categoria}.\n"
        f"Mensagem do cliente: {msg}\n"
        "Responda de forma direta em 1 frase."
    )
    resp = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    return resp.choices[0].message.content

O `for` aqui simula uma fila de atendimento. Em produção, essas mensagens viriam de um banco de dados, de uma fila como o Kafka, ou de um webhook do WhatsApp.

In [ ]:
# for — processa uma fila de mensagens
mensagens = [
    "quero cancelar minha assinatura",
    "minha fatura veio errada",
    "o sistema travou ao abrir"
]

for msg in mensagens:
    categoria = classificar(msg)
    resposta = responder(msg, categoria)
    print(f"[{categoria}] {resposta}")
    print()

[RETENCAO] Entendo que você deseja cancelar sua assinatura, mas gostaria de saber se há algo que possamos fazer para melhorar sua experiência e mudar sua decisão antes de prosseguirmos com o cancelamento.

[FINANCEIRO] Vamos verificar o problema com a sua fatura juntos, por favor, envie mais detalhes sobre o erro para que eu possa ajudá-lo a resolver essa questão o mais rápido possível.

[SUPORTE] Lamento pelo inconveniente, por favor, tente reiniciar o sistema e verifique se o problema persiste, caso contrário, estou aqui para ajudá-lo a encontrar uma solução.



---
## 4. Listas e dicionários — o coração das APIs de IA

### Conceito
LLMs como o Llama e o GPT não têm memória nativa entre mensagens. A "memória" de um chatbot é uma **lista de dicionários** que você monta e envia completa a cada chamada. O modelo lê tudo do zero e simula continuidade.

### Estrutura obrigatória da API
Toda chamada a qualquer API de LLM (Groq, OpenAI, Anthropic) exige o campo `messages` com essa estrutura exata:
```python
[
    {"role": "system",    "content": "instruções do sistema"},
    {"role": "user",      "content": "mensagem do usuário"},
    {"role": "assistant", "content": "resposta anterior do modelo"}
]
```

### Técnico
- **`role: system`** — define a personalidade e as regras do agente. É lido primeiro e tem o maior peso.
- **`role: user`** — mensagem do humano.
- **`role: assistant`** — resposta anterior do modelo. Incluir isso é o que dá "memória" ao chat.
- A cada novo turno, você faz `.append()` na lista e reenvia tudo.

In [ ]:
# list — historico cresce a cada turno da conversa
historico = [
    {"role": "system", "content": "Voce e o SupportBot da empresa. Responda em 1 frase."}
]

def conversar(mensagem_usuario):
    # dict — estrutura exata que a API espera
    historico.append({"role": "user", "content": mensagem_usuario})

    resposta = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=historico  # passa a lista inteira — o modelo ve o contexto completo
    )

    texto = resposta.choices[0].message.content
    historico.append({"role": "assistant", "content": texto})
    return texto

Repare que no segundo turno o modelo **lembra** da fatura mencionada no primeiro. Isso acontece porque a lista `historico` já contém a primeira troca quando fazemos a segunda chamada.

In [ ]:
# Simulando 2 turnos — o modelo lembra do primeiro no segundo
print("Turno 1:", conversar("Fui cobrado errado na fatura"))
print()
print("Turno 2:", conversar("E se eu quiser cancelar por causa disso?"))

Turno 1: Peço desculpas pelo inconveniente, gostaria de saber mais sobre o erro na fatura para que possamos investigar e corrigir o problema o mais rápido possível.

Turno 2: Se você deseja cancelar o serviço devido ao erro na fatura, posso ajudá-lo a iniciar o processo de cancelamento e também verificar se há alguma forma de compensar o erro, como um crédito ou reembolso.


In [ ]:
# Inspecionando o historico acumulado — veja como a lista cresceu
print(f"Total de mensagens no historico: {len(historico)}\n")
for msg in historico:
    print(f"[{msg['role']:10}] {msg['content'][:70]}")

Total de mensagens no historico: 5

[system    ] Voce e o SupportBot da empresa. Responda em 1 frase.
[user      ] Fui cobrado errado na fatura
[assistant ] Peço desculpas pelo inconveniente, gostaria de saber mais sobre o erro
[user      ] E se eu quiser cancelar por causa disso?
[assistant ] Se você deseja cancelar o serviço devido ao erro na fatura, posso ajud


---
## 5. Tokens e temperatura

### O que são tokens?
Modelos de linguagem não leem palavras — leem **tokens**, que são fragmentos de texto. Em inglês, 1 token ≈ 4 caracteres. Em português, por ter palavras mais longas e acentuação, 1 token ≈ 3 caracteres.

**Por que importa?**
- APIs cobram por token (input + output)
- Cada modelo tem um limite de tokens por chamada (janela de contexto): Llama 3.3 70B suporta até 128k tokens
- Prompts muito longos aumentam o custo e podem ultrapassar o limite

### O que é temperatura?
Controla o quão "criativo" ou "determinístico" o modelo é:

| Valor | Comportamento | Use para |
|-------|--------------|----------|
| `0.0` | Sempre a mesma resposta | Classificação, roteamento |
| `0.3–0.5` | Pequenas variações | Respostas técnicas |
| `0.7–0.9` | Criativo e variado | Emails, textos persuasivos |
| `1.2+` | Caótico, imprevisível | Evitar em produção |

### Visualizando tokens com o tokenizer oficial

O código abaixo carrega o tokenizer do Llama 3.3 e mostra como ele divide um texto em tokens. Isso deixa claro por que palavras em português custam mais tokens que em inglês.

In [ ]:
# pip install tokenizers  (descomentar se necessário)
# !pip install tokenizers

from tokenizers import Tokenizer

# Tokenizer oficial do LLaMA 3 (mesmo usado pelo Groq)
t = Tokenizer.from_pretrained("meta-llama/Meta-Llama-3-70B-Instruct")

texto_pt = "Estamos revisando a semana 02 da imersão de IA da Alura e Santander"
texto_en = "We are reviewing week 02 of the AI immersion from Alura and Santander"

out_pt = t.encode(texto_pt)
out_en = t.encode(texto_en)

print("=== Português ===")
print(f"Tokens: {out_pt.tokens}")
print(f"Quantidade: {len(out_pt.ids)} tokens")

print("\n=== Inglês ===")
print(f"Tokens: {out_en.tokens}")
print(f"Quantidade: {len(out_en.ids)} tokens")

print(f"\nO mesmo texto em PT usa {len(out_pt.ids) - len(out_en.ids)} tokens a mais que em EN")

=== Português ===
Tokens: ['<|begin_of_text|>', 'Est', 'amos', 'Ġrevis', 'ando', 'Ġa', 'Ġsemana', 'Ġ', '02', 'Ġda', 'Ġim', 'ers', 'Ã£o', 'Ġde', 'ĠIA', 'Ġda', 'ĠAl', 'ura', 'Ġe', 'ĠSant', 'ander']
Quantidade: 21 tokens

=== Inglês ===
Tokens: ['<|begin_of_text|>', 'We', 'Ġare', 'Ġreviewing', 'Ġweek', 'Ġ', '02', 'Ġof', 'Ġthe', 'ĠAI', 'Ġimmersion', 'Ġfrom', 'ĠAl', 'ura', 'Ġand', 'ĠSant', 'ander']
Quantidade: 17 tokens

O mesmo texto em PT usa 4 tokens a mais que em EN


### Demonstração ao vivo: o mesmo prompt com temperaturas diferentes

Veja como o modelo responde de forma idêntica em `0.0` e cada vez diferente em `1.5`. Isso torna a escolha de temperatura uma decisão de engenharia, não de preferência.

In [ ]:
prompt = "O cliente quer cancelar. Ofereça um desconto para retê-lo. Responda em 1 frase."

temperaturas = [0.0, 0.7, 1.5]

for temp in temperaturas:
    print(f"\n{'='*50}")
    print(f"temperature = {temp}")
    print('='*50)
    for i in range(1, 3):
        resp = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=temp
        )
        print(f"  Chamada {i}: {resp.choices[0].message.content}")


temperature = 0.0
  Chamada 1: "Entendo que você está considerando cancelar, mas gostaria de oferecer um desconto especial de 20% em sua próxima compra para tentar retê-lo como cliente."
  Chamada 2: "Entendo que você está considerando cancelar, mas gostaria de oferecer um desconto especial de 20% em sua próxima compra para tentar retê-lo como cliente."

temperature = 0.7
  Chamada 1: "Eu posso oferecer um desconto especial de [porcentagem ou valor] para você permanecer como nosso cliente, é o que você acha?"
  Chamada 2: "Se cancelar o contrato, estou disposto a oferecer um desconto de 10% na próxima compra para tentar retê-lo como cliente."

temperature = 1.5
  Chamada 1: Posso oferecer um desconto especial de 10% em sua compra atual, além de uma recompensa futura, desde que decida reassinar ou não cancelar sua subscrição.
  Chamada 2: Quero oferecer um desconto adicional de até 20% em seus próximos produtos para que você não parta sem experimentar mais as nossas excelentes ofertas.

---
## 6. Engenharia de Prompt — 4 técnicas fundamentais

Engenharia de prompt é a prática de **estruturar a instrução** que você dá ao modelo para obter respostas melhores, mais confiáveis e mais controláveis.

Não é "escrever bem" — é conhecer os padrões que mudam o comportamento do modelo de forma previsível.

### As 4 técnicas que vamos ver

| Técnica | O que muda | Quando usar |
|---------|-----------|-------------|
| **Few-Shot** | O modelo imita os exemplos que você deu | Classificação, formatação consistente |
| **Chain-of-Thought** | O modelo raciocina antes de responder | Lógica, cálculos, decisões complexas |
| **Chain of Verification** | O modelo critica e corrige a própria resposta | Reduzir alucinações, dados críticos |
| **Self-Consistency** | Várias chamadas votam na melhor resposta | Decisões irreversíveis, alta confiança |

Vamos usar o **mesmo caso** para todas as técnicas, para que você possa comparar os resultados.

In [ ]:
CASO = "Cliente comprou há 8 dias, produto com defeito leve, sem nota fiscal."

def chamar(prompt, temp=0.0):
    resp = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=temp
    )
    return resp.choices[0].message.content.strip()

print(f"Caso de teste: {CASO}")

---
### 6.1 Few-Shot Prompting — ensine pelo exemplo

**Conceito:** Você inclui exemplos já resolvidos no prompt antes de fazer a pergunta real. O modelo aprende o padrão dos exemplos e aplica na nova situação.

**Por que funciona:** LLMs são treinados para completar padrões. Quando você mostra 3 exemplos com a mesma estrutura, o modelo entende implicitamente o formato esperado e a lógica de decisão.

**Técnico:** Os exemplos ficam no próprio texto do prompt, não em parâmetros separados. Quanto mais representativos forem os exemplos, melhor o modelo generaliza para casos novos. O ideal é incluir exemplos de cada classe (positivo e negativo) e casos de borda.

In [ ]:
print("="*55)
print("1. FEW-SHOT PROMPTING")
print("="*55)

prompt_fewshot = f"""
Classifique se devemos REEMBOLSAR ou NEGAR com base nos exemplos:

Exemplo 1: "Produto com defeito grave, 2 dias, com NF." → REEMBOLSAR
Exemplo 2: "Arrependimento, 40 dias, sem defeito." → NEGAR
Exemplo 3: "Produto errado enviado, 5 dias, com NF." → REEMBOLSAR

Agora classifique (responda só REEMBOLSAR ou NEGAR):
Caso: "{CASO}"
"""
print(chamar(prompt_fewshot))

---
### 6.2 Chain-of-Thought (CoT) — force o raciocínio passo a passo

**Conceito:** Em vez de pedir a resposta direta, você pede que o modelo "pense em voz alta" antes de concluir. Isso força uma análise estruturada e torna os erros de raciocínio visíveis.

**Por que funciona:** Modelos menores frequentemente erram ao tentar responder diretamente problemas complexos — eles "pulam" etapas. Quando obrigados a escrever cada etapa, o modelo se autocorrige durante o processo.

**Técnico:** A frase mágica é `"Pense passo a passo"` ou estruturar numericamente as etapas de análise. Isso aumenta o número de tokens gerados, o que eleva o custo — mas também a qualidade da resposta em tarefas complexas. Compare a saída desta técnica com a do Few-Shot acima.

In [ ]:
print("="*55)
print("2. CHAIN-OF-THOUGHT (CoT)")
print("="*55)

prompt_cot = f"""
Analise passo a passo e decida se deve REEMBOLSAR ou NEGAR.

Caso: "{CASO}"

Pense passo a passo:
1. Qual o prazo? Está dentro da política de 7 dias?
2. Há defeito? Qual a gravidade?
3. Há nota fiscal? Como isso afeta a decisão?
4. DECISÃO FINAL: REEMBOLSAR ou NEGAR?
"""
print(chamar(prompt_cot))

---
### 6.3 Chain of Verification (CoVe) — o modelo corrige a si mesmo

**Conceito:** Técnica de 3 passos onde o modelo (1) gera uma resposta, (2) formula perguntas para verificar as afirmações que fez, e (3) responde essas perguntas e corrige a resposta se necessário.

**Por que funciona:** Modelos de linguagem "alucinam" porque são otimizados para gerar texto coerente, não necessariamente verdadeiro. Ao forçar a verificação explícita, o modelo tem a chance de detectar inconsistências antes de entregar a resposta final.

**Técnico:** O CoVe é uma forma de **prompt chaining**: a saída do Passo 1 alimenta o Passo 2, que alimenta o Passo 3. Em sistemas mais sofisticados, cada passo pode ser uma chamada separada para um modelo diferente. Aqui fazemos tudo em uma única chamada por simplicidade.

In [ ]:
print("="*55)
print("3. CHAIN OF VERIFICATION (CoVe)")
print("="*55)

prompt_cove = f"""
Caso: "{CASO}"

PASSO 1 — Gere uma resposta inicial sobre reembolso (REEMBOLSAR ou NEGAR com justificativa breve).

PASSO 2 — Liste 2 afirmações da sua resposta e formule uma pergunta de verificação para cada.

PASSO 3 — Responda cada pergunta e, se necessário, corrija sua decisão final.
"""
print(chamar(prompt_cove))

---
### 6.4 Self-Consistency — votação entre múltiplas respostas

**Conceito:** Envia o mesmo prompt várias vezes com temperatura alta (modelo mais criativo/variável) e usa a resposta que aparece com mais frequência como decisão final. É uma votação por maioria.

**Por que funciona:** Com `temperature > 0`, o modelo não é determinístico — cada chamada explora um "caminho de raciocínio" diferente. Se a maioria dos caminhos chega à mesma conclusão, há uma evidência forte de que aquela é a resposta correta.

**Técnico:**
- Use `Counter` do módulo `collections` para contar frequências
- Calcule confiança: `freq / total_de_chamadas`
- Defina um limiar (ex: 60%): abaixo disso, o modelo está inseguro → escale para humano
- Custo: N chamadas = N × custo individual. Use com moderação, apenas em decisões críticas.

> Observe a saída: às vezes o modelo discorda de si mesmo entre as 5 chamadas. Esse comportamento ao vivo é o melhor argumento para a técnica existir.

In [ ]:
from collections import Counter

print("="*55)
print("4. SELF-CONSISTENCY (5 chamadas, temp=0.8)")
print("="*55)

prompt_sc = f"""
Caso: "{CASO}"
Decisão (responda SOMENTE com REEMBOLSAR ou NEGAR):
"""

respostas = [chamar(prompt_sc, temp=0.8) for _ in range(5)]
print("Respostas individuais:", respostas)

consenso, freq = Counter(respostas).most_common(1)[0]
confianca = freq / len(respostas)
print(f"\nConsenso: {consenso}")
print(f"Confiança: {confianca:.0%}")

if confianca < 0.6:
    print("→ Confiança baixa — escalar para humano")
else:
    print(f"→ Decisão automática: {consenso}")

---
## ✅ Resumo da Semana 02

Cada conceito Python que vimos hoje é uma peça de um agente de IA real:

| Conceito Python | Papel no agente |
|----------------|----------------|
| `type()` | Navegar objetos de resposta da API |
| `f-string` | Montar prompts dinâmicos (= PromptTemplate) |
| `if/elif/else` | Roteador de intenção |
| `for/while` | Processar filas de mensagens |
| `list` + `dict` | Histórico de conversa (= Chat Memory) |
| `try/except` | Retry com backoff exponencial |
| `Counter` | Self-Consistency / votação |
| `temperature` | Controle criatividade vs. determinismo |

> **Próxima semana:** funções, classes e módulos — as estruturas que organizam todo esse código em um agente de verdade.